<a href="https://colab.research.google.com/github/KalerKaler/PlaylistMaker/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [3]:
!apt install python3-libtorrent
!pip install --force-reinstall libtorrent

import libtorrent as lt

ses = lt.session()
ses.listen_on(6881, 6891)
downloads = []

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-libtorrent is already the newest version (2.0.5-5).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 30.3 MB/s eta 0:00:00
  Attempting uninstall: libtorrent
    Found existing installation: libtorrent 2.0.5-build-libtorrent-rasterbar-qrM5vM-libtorrent-rasterbar-2.0.5-bindings-python
    Can't uninstall 'libtorrent'. No files were found to uninstall.


/tmp/ipykernel_937/254432028.py:7: DeprecationWarning: listen_on() is deprecated
  ses.listen_on(6881, 6891)


### Mount Google Drive
To stream files we need to mount Google Drive.

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Add From Torrent File
You can run this cell to add more files as many times as you want

In [5]:
from google.colab import files

source = files.upload()
params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))

IndexError: list index out of range

### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [7]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )


Enter Magnet Link Or Type Exit: magnet:?xt=urn:btih:A4E95E8C10CC6104C06662B0A2A3109AD3A4A995&dn=Forza+Horizon+6+%28v354.221+%2B+10+DLCs+%2B+Multiplayer%2C+MULTi23%29+ %5BFitGirl+Repack%2C+Selective+Download+-+from+89+GB%5D&tr=udp%3A%2F%2Fopentor.net%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce&tr=udp%3A%2F%2Ftracker.torrent.eu.org%3A451%2Fannounce&tr=udp%3A%2F%2Ftracker.qu.ax%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.fnix.n et%3A6969%2Fannounce&tr=udp%3A%2F%2Fevan.im%3A6969%2Fannounce&tr=udp%3A%2F%2Fmartin-gebhardt.eu%3A25%2Fannounce&tr=https%3A%2F%2Fshahidrazi.online%3A443%2Fannounce&tr=http%3A%2F%2Fwegkxfcivgx.ydns.eu%3A80%2Fannounce&tr=http%3A%2F%2Flucke.fenesisu.moe%3A6969%2Fannounce&tr=udp%3A%2F%2Fextracker.dahrkael.net%3A6969%2Fannounce&tr=https%3A%2F%2Ftracker.alaskantf.com%3A443%2Fannounce&tr=https%3A%2F%2Ftracker.qingwa.pro%3A443%2Fannounce&tr=udp%3A%2F%2Ftracker.playground.ru%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce

/tmp/ipykernel_937/548325381.py:8: DeprecationWarning: add_magnet_uri() is deprecated
  lt.add_magnet_uri(ses, magnet_link, params)


Enter Magnet Link Or Type Exit: Exit


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            bar.description = " ".join(
                [
                    download.name(),
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close() # Seems to be not working in Colab (see https://github.com/googlecolab/colabtools/issues/726#issue-486731758)
            download_bars.remove(bar)
            print(download.name(), "complete")
    time.sleep(1)


FloatSlider(value=0.0, disabled=True, layout=Layout(width='auto'), step=0.01, style=SliderStyle(description_wi…

/tmp/ipykernel_937/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_937/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_937/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_937/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_937/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_937/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_937/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_937/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_937/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_937/3737971044.py:35: DeprecationWarning: name(